# 5SC28 Design Assignment — Section 4.2.2: Reference Tracking with Actor-Critic

This notebook extends the A2C swing-up policy to learn a **single policy** that can:
1. Swing up the disc from the bottom ($\theta = 0$).
2. Track a reference $\theta_\mathrm{ref} = \pi + \delta$, where $\delta \in [-15°, +15°]$.

> **Note**: a switching controller combining multiple single-target policies is **not valid**.
> This is a single end-to-end trained network.

**Approach**: augment the observation with $[\sin\theta_\mathrm{ref},\,\cos\theta_\mathrm{ref}]$.
The reference is sampled randomly at each episode reset. Reward:
$$r = \exp\!\left(-\frac{\angle(\theta - \theta_\mathrm{ref})^2}{2(\pi/8)^2}\right)$$

In [ ]:
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal
import gymnasium as gym
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

sys.path.insert(0, '/Users/cezar/Documents/Documents_mac/University/TUe/Q6/gym-unbalanced-disk')
import gym_unbalanced_disk  # registers 'unbalanced-disk-sincos-v0'

## Reference-Tracking Environment Wrapper

In [ ]:
class UnbalancedDiskRef(gym.Wrapper):
    """
    Observation (5-D): [sin(th), cos(th), omega, sin(th_ref), cos(th_ref)]
    Reward: exp( -angle_err^2 / (2 * SIGMA_R^2) )
    """
    REF_RANGE = np.pi / 12   # +-15 degrees
    SIGMA_R   = np.pi / 8    # reward width ~22.5 degrees

    def __init__(self, env):
        super().__init__(env)
        low  = np.concatenate([env.observation_space.low,  [-1., -1.]])
        high = np.concatenate([env.observation_space.high, [ 1.,  1.]])
        self.observation_space = gym.spaces.Box(
            low=low.astype(np.float32), high=high.astype(np.float32))
        self.th_ref = np.pi

    def reset(self, **kwargs):
        obs, info   = self.env.reset(**kwargs)
        self.th_ref = np.pi + np.random.uniform(-self.REF_RANGE, self.REF_RANGE)
        return self._aug(obs), info

    def step(self, action):
        obs, _, terminated, truncated, info = self.env.step(action)
        th_est = np.arctan2(obs[0], obs[1])
        th_err = np.arctan2(np.sin(th_est - self.th_ref),
                            np.cos(th_est - self.th_ref))
        reward = float(np.exp(-th_err**2 / (2 * self.SIGMA_R**2)))
        return self._aug(obs), reward, terminated, truncated, info

    def _aug(self, obs):
        return np.concatenate([obs, [np.sin(self.th_ref), np.cos(self.th_ref)]])


env_ref = UnbalancedDiskRef(gym.make('unbalanced-disk-sincos-v0'))
obs, _  = env_ref.reset()
print(f'Observation dim : {obs.shape}')  # (5,)
print(f'th_ref          : {np.degrees(env_ref.th_ref):.1f} deg')
print(f'Initial obs     : {obs}')

## Actor-Critic Network

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim: int, hidden_size: int = 64, umax: float = 3.0):
        super().__init__()
        self.umax = umax
        self.critic_l1 = nn.Linear(obs_dim, hidden_size)
        self.critic_l2 = nn.Linear(hidden_size, 1)
        self.actor_l1  = nn.Linear(obs_dim, hidden_size)
        self.actor_mu  = nn.Linear(hidden_size, 1)
        self.log_sigma = nn.Parameter(torch.tensor([0.0]))

    def critic(self, state):
        h = torch.tanh(self.critic_l1(state))
        return self.critic_l2(h)[:, 0]

    def actor(self, state):
        h     = torch.tanh(self.actor_l1(state))
        mu    = self.umax * torch.tanh(self.actor_mu(h)[:, 0])
        sigma = self.log_sigma.exp().expand_as(mu)
        return mu, sigma

## Helper Functions

In [ ]:
def rollout(actor_crit, env, N_rollout=10_000):
    Start_state, Actions, Rewards, End_state, Terminal = [], [], [], [], []
    obs, _ = env.reset()
    with torch.no_grad():
        for _ in range(N_rollout):
            state_t   = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            mu, sigma = actor_crit.actor(state_t)
            action    = float(np.clip(Normal(mu, sigma).sample().item(),
                                      -actor_crit.umax, actor_crit.umax))
            next_obs, reward, terminated, truncated, _ = env.step(np.float32(action))
            Start_state.append(obs);  Actions.append(action)
            Rewards.append(reward);   End_state.append(next_obs)
            Terminal.append(terminated)
            obs = next_obs
            if terminated or truncated:
                obs, _ = env.reset()
    return (np.array(Start_state),
            np.array(Actions,  dtype=np.float32),
            np.array(Rewards,  dtype=np.float32),
            np.array(End_state),
            np.array(Terminal, dtype=np.int32))


def eval_actor(actor_crit, env, n_episodes=5):
    total = 0.0
    with torch.no_grad():
        for _ in range(n_episodes):
            obs, _ = env.reset()
            ep_r   = 0.0
            while True:
                state_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                mu, _   = actor_crit.actor(state_t)
                obs, r, terminated, truncated, _ = env.step(np.float32(mu.item()))
                ep_r += r
                if terminated or truncated:
                    break
            total += ep_r
    return total / n_episodes


def A2C_train(actor_crit, optimizer, env,
              alpha_actor=1.0, alpha_entropy=0.02, gamma=0.98,
              N_iterations=50, N_rollout=10_000, N_epochs=5,
              batch_size=64, N_evals=5, checkpoint='ac_checkpoint.pt'):
    best = -float('inf');  score_hist = []
    torch.save(actor_crit.state_dict(), checkpoint)
    try:
        for iteration in tqdm(range(N_iterations), desc='A2C'):
            S, A, R, S_next, T = rollout(actor_crit, env, N_rollout)
            S_t     = torch.tensor(S,      dtype=torch.float32)
            A_t     = torch.tensor(A,      dtype=torch.float32)
            R_t     = torch.tensor(R,      dtype=torch.float32)
            Snext_t = torch.tensor(S_next, dtype=torch.float32)
            T_t     = torch.tensor(T,      dtype=torch.float32)
            for _ in range(N_epochs):
                idx = torch.randperm(len(S_t))
                for i in range(batch_size, len(S_t) + 1, batch_size):
                    b  = idx[i - batch_size : i]
                    sb, ab, rb = S_t[b], A_t[b], R_t[b]
                    snb, tb    = Snext_t[b], T_t[b]
                    adv  = rb + gamma * actor_crit.critic(snb) * (1.0 - tb) \
                              - actor_crit.critic(sb)
                    mu, sigma = actor_crit.actor(sb)
                    dist      = Normal(mu, sigma)
                    L_vf  = (adv ** 2).mean()
                    L_pg  = -(adv.detach() * dist.log_prob(ab)).mean()
                    L_ent = -dist.entropy().mean()
                    loss  = L_vf + alpha_actor * L_pg + alpha_entropy * L_ent
                    optimizer.zero_grad();  loss.backward()
                    nn.utils.clip_grad_norm_(actor_crit.parameters(), 1.0)
                    optimizer.step()
            score = eval_actor(actor_crit, env, N_evals)
            score_hist.append(score)
            tqdm.write(f'iter {iteration:3d}  sigma={actor_crit.log_sigma.exp().item():.3f}  score={score:.1f}')
            if score > best:
                best = score
                torch.save(actor_crit.state_dict(), checkpoint)
                tqdm.write(f'  --> new best {best:.1f}, checkpoint saved')
    finally:
        actor_crit.load_state_dict(torch.load(checkpoint))
        print(f'Loaded best checkpoint (score = {best:.1f})')
    return score_hist

## Training

In [ ]:
env_ref     = UnbalancedDiskRef(gym.make('unbalanced-disk-sincos-v0'))
obs_dim_ref = env_ref.observation_space.shape[0]  # 5

torch.manual_seed(0)
actor_crit_ref = ActorCritic(obs_dim=obs_dim_ref, hidden_size=64, umax=3.0)
optimizer_ref  = torch.optim.Adam(actor_crit_ref.parameters(), lr=5e-4)

score_hist_ref = A2C_train(
    actor_crit_ref, optimizer_ref, env_ref,
    alpha_actor=1.0, alpha_entropy=0.02, gamma=0.98,
    N_iterations=50, N_rollout=10_000, N_epochs=5,
    batch_size=64, N_evals=5, checkpoint='ac_reftrack.pt',
)

plt.figure(figsize=(8, 3))
plt.plot(score_hist_ref, 'b-o', ms=3)
plt.xlabel('Iteration');  plt.ylabel('Average episode reward')
plt.title('A2C - Reference Tracking')
plt.grid(True);  plt.tight_layout();  plt.show()

## Evaluation Across Reference Angles

In [ ]:
def eval_at_ref(actor_crit, ref_deg, n_episodes=5):
    th_ref = np.pi + np.radians(ref_deg)
    env_r  = UnbalancedDiskRef(gym.make('unbalanced-disk-sincos-v0'))
    total  = 0.0
    with torch.no_grad():
        for _ in range(n_episodes):
            obs, _       = env_r.reset()
            env_r.th_ref = th_ref
            obs[-2]      = np.sin(th_ref)
            obs[-1]      = np.cos(th_ref)
            ep_r         = 0.0
            while True:
                state_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                mu, _   = actor_crit.actor(state_t)
                obs, r, terminated, truncated, _ = env_r.step(np.float32(mu.item()))
                ep_r   += r
                if terminated or truncated:
                    break
            total += ep_r
    return total / n_episodes


refs           = [-15, -10, -5, 0, 5, 10, 15]
scores_per_ref = {}
for ref in refs:
    s = eval_at_ref(actor_crit_ref, ref, n_episodes=5)
    scores_per_ref[ref] = s
    print(f'ref = {ref:+3d} deg   avg reward = {s:.1f}')

plt.figure(figsize=(6, 3))
plt.bar([f'{r:+d} deg' for r in refs], [scores_per_ref[r] for r in refs], color='steelblue')
plt.xlabel('Reference offset from top');  plt.ylabel('Avg episode reward')
plt.title('Reference-tracking performance')
plt.tight_layout();  plt.show()

## Episode Plots at Three Reference Angles

In [ ]:
def plot_ref_episode(actor_crit, ref_deg):
    th_ref  = np.pi + np.radians(ref_deg)
    env_r   = UnbalancedDiskRef(gym.make('unbalanced-disk-sincos-v0'))
    obs, _  = env_r.reset()
    env_r.th_ref = th_ref
    obs[-2] = np.sin(th_ref)
    obs[-1] = np.cos(th_ref)

    obs_list, act_list, rew_list = [], [], []
    with torch.no_grad():
        while True:
            state_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            mu, _   = actor_crit.actor(state_t)
            obs_list.append(obs.copy())
            act_list.append(mu.item())
            obs, r, terminated, truncated, _ = env_r.step(np.float32(mu.item()))
            rew_list.append(r)
            if terminated or truncated:
                break

    obs_arr = np.array(obs_list)
    th_arr  = np.degrees(np.arctan2(obs_arr[:, 0], obs_arr[:, 1]))
    t       = np.arange(len(th_arr)) * 0.025

    fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
    axes[0].plot(t, th_arr, 'b')
    axes[0].axhline(180, color='r', ls='--', alpha=0.5, label='top (180 deg)')
    axes[0].axhline(180 + ref_deg, color='orange', ls='--',
                    label=f'reference ({180+ref_deg:.0f} deg)')
    axes[0].set_ylabel('theta [deg]');  axes[0].legend();  axes[0].grid(True)
    axes[1].plot(t, obs_arr[:, 2], 'g')
    axes[1].set_ylabel('omega [rad/s]');  axes[1].grid(True)
    axes[2].plot(t[:len(act_list)], act_list, 'r')
    axes[2].axhline( 3, color='k', ls=':', alpha=0.4, label='+-3 V')
    axes[2].axhline(-3, color='k', ls=':', alpha=0.4)
    axes[2].set_ylabel('u [V]');  axes[2].set_xlabel('time [s]')
    axes[2].legend();  axes[2].grid(True)
    plt.suptitle(f'Reference tracking  ref = {180+ref_deg:.0f} deg  '
                 f'(total reward = {sum(rew_list):.1f})')
    plt.tight_layout();  plt.show()


for ref_deg in [-15, 0, 15]:
    plot_ref_episode(actor_crit_ref, ref_deg)